# ============================================
#  Notebook 05 — Feature Extraction, OCR Quality & BERT
#  Memorial Sloan Kettering | Goel Lab
# ============================================

# Notebook 5: Feature Extraction — OCR Quality Scoring, BERT Embeddings & Feature Interaction Analysis

**Purpose:**
1. Per-page OCR image quality metrics (Laplacian variance, Tenengrad, contrast, etc.)
2. BERT-based document embeddings for extracted text
3. Document-level feature engineering (token counts, lexical diversity, negation frequency, etc.)
4. H2O-based feature interaction analysis for AI accuracy/fabrication prediction

**Templates:**
- BERT encoder: `copy_of_bert_end_to_end_...py`
- Feature interaction: `ai_feature_interaction_analysis.py`

In [ ]:
import os
import re
import json
from pathlib import Path
from typing import Dict, List
from dotenv import load_dotenv

import fitz  # PyMuPDF
import pytesseract
import pandas as pd
import numpy as np
import cv2
from PIL import Image
import matplotlib.pyplot as plt
import seaborn as sns
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

sns.set_style('whitegrid')

load_dotenv()

# ── Paths ─────────────────────────────────────────────────────────────────────
PROJECT_ROOT     = Path(os.getenv("PROJECT_ROOT",
    r"C:\Users\jamesr4\OneDrive - Memorial Sloan Kettering Cancer Center\Documents\GitHub\llm_summarization_br_ca"))
DATA_PRIVATE_DIR = Path(os.getenv("DATA_PRIVATE_DIR", r"C:\Users\jamesr4\loc\data_private"))

RAW_DIR     = DATA_PRIVATE_DIR / "raw"           # source PDFs with PHI
TEXT_DIR    = DATA_PRIVATE_DIR / "extracted_text" # per-case deidentified text
OUTPUT_DIR  = PROJECT_ROOT / "reports"           # figures and tables
FEATURE_DIR = PROJECT_ROOT / "data" / "features" # non-PHI feature CSVs (in repo)

FEATURE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

---
## Part 1: OCR Image Quality Scoring

Per-page metrics computed from rasterized PDF pages:
- **Variance of Laplacian** (blur detection)
- **Tenengrad** (gradient energy)
- **RMS contrast**
- **Intensity spread** (p95 - p5)
- **Mean brightness**
- **Skew angle** (via Hough transform)
- **Resolution / DPI**

In [ ]:
def render_page_to_cv(doc, page_index, dpi=300):
    page = doc.load_page(page_index)
    zoom = dpi / 72.0
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    img = np.frombuffer(pix.samples, dtype=np.uint8).reshape(pix.height, pix.width, 3)
    return cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), pix.width, pix.height

def laplacian_variance(gray):
    return cv2.Laplacian(gray, cv2.CV_64F).var()

def tenengrad(gray):
    gx = cv2.Sobel(gray, cv2.CV_64F, 1, 0, ksize=3)
    gy = cv2.Sobel(gray, cv2.CV_64F, 0, 1, ksize=3)
    return np.sqrt(gx**2 + gy**2).mean()

def rms_contrast(gray):
    return float(gray.astype(float).std())

def intensity_spread(gray):
    p5 = np.percentile(gray, 5)
    p95 = np.percentile(gray, 95)
    return float(p95 - p5)

def mean_brightness(gray):
    return float(gray.mean())

def estimate_skew_angle(gray):
    """Estimate skew angle using Hough line transform."""
    edges = cv2.Canny(gray, 50, 150, apertureSize=3)
    lines = cv2.HoughLines(edges, 1, np.pi / 180, threshold=200)
    if lines is None or len(lines) == 0:
        return 0.0
    angles = []
    for rho, theta in lines[:, 0]:
        angle = (theta * 180 / np.pi) - 90
        if -45 < angle < 45:
            angles.append(angle)
    return float(np.median(angles)) if angles else 0.0

def compute_page_quality(gray, width, height, dpi=300):
    return {
        "laplacian_var": laplacian_variance(gray),
        "tenengrad": tenengrad(gray),
        "rms_contrast": rms_contrast(gray),
        "intensity_spread_p95_p5": intensity_spread(gray),
        "mean_brightness": mean_brightness(gray),
        "skew_angle_deg": estimate_skew_angle(gray),
        "dpi": dpi,
        "width_px": width,
        "height_px": height,
    }

print("Image quality functions defined.")

In [ ]:
# Run quality scoring on all PDFs
import hashlib
def generate_case_id(filename: str) -> str:
    h = hashlib.sha256(filename.encode()).hexdigest()[:12]
    return f"CASE_{h.upper()}"

quality_rows = []
pdf_paths = sorted(RAW_DIR.rglob("*.pdf"))

for pdf_path in tqdm(pdf_paths, desc="Scoring image quality"):
    case_id = generate_case_id(pdf_path.stem)
    try:
        doc = fitz.open(pdf_path)
    except Exception as e:
        quality_rows.append({"case_id": case_id, "file": pdf_path.name, "status": "ERROR", "error": str(e)})
        continue
    for p in range(doc.page_count):
        gray, w, h = render_page_to_cv(doc, p)
        metrics = compute_page_quality(gray, w, h)
        metrics["case_id"] = case_id
        metrics["file"] = pdf_path.name
        metrics["page"] = p + 1
        quality_rows.append(metrics)
    doc.close()

df_quality = pd.DataFrame(quality_rows)
df_quality.to_csv(FEATURE_DIR / "page_level_ocr_quality.csv", index=False)
print(f"Page-level quality metrics: {df_quality.shape}")
df_quality.head()

In [ ]:
# Flag pages using percentile thresholds
if "laplacian_var" in df_quality.columns and not df_quality["laplacian_var"].isna().all():
    blur_thresh = df_quality["laplacian_var"].quantile(0.10)
    contrast_thresh = df_quality["rms_contrast"].quantile(0.10)
    df_quality["flag_blurry"] = df_quality["laplacian_var"] < blur_thresh
    df_quality["flag_low_contrast"] = df_quality["rms_contrast"] < contrast_thresh

    print(f"Blur threshold (10th pctile): {blur_thresh:.2f}")
    print(f"Contrast threshold (10th pctile): {contrast_thresh:.2f}")
    print(f"Pages flagged blurry: {df_quality['flag_blurry'].sum()}")
    print(f"Pages flagged low contrast: {df_quality['flag_low_contrast'].sum()}")
else:
    print("No quality data available (no PDFs found).")

# Case-level summary
case_quality = (
    df_quality.groupby("case_id")
    .agg(
        num_pages=("page", "count"),
        avg_laplacian_var=("laplacian_var", "mean"),
        worst_laplacian_var=("laplacian_var", "min"),
        avg_rms_contrast=("rms_contrast", "mean"),
        pct_blurry=("flag_blurry", "mean") if "flag_blurry" in df_quality.columns else ("page", "count"),
        pct_low_contrast=("flag_low_contrast", "mean") if "flag_low_contrast" in df_quality.columns else ("page", "count"),
    )
    .reset_index()
)
case_quality.to_csv(FEATURE_DIR / "case_level_ocr_quality.csv", index=False)
print(f"\nCase-level quality summary: {case_quality.shape}")
case_quality.head()

In [ ]:
# Quality visualization
if len(df_quality) > 0 and "laplacian_var" in df_quality.columns:
    fig, axes = plt.subplots(2, 3, figsize=(16, 9))
    metrics_to_plot = ["laplacian_var", "tenengrad", "rms_contrast",
                       "intensity_spread_p95_p5", "mean_brightness", "skew_angle_deg"]
    titles = ["Laplacian Variance (blur)", "Tenengrad (sharpness)", "RMS Contrast",
              "Intensity Spread (p95-p5)", "Mean Brightness", "Skew Angle (deg)"]
    for ax, metric, title in zip(axes.flatten(), metrics_to_plot, titles):
        vals = df_quality[metric].dropna()
        if len(vals) > 0:
            ax.hist(vals, bins=30, color="#3498db", edgecolor="black", alpha=0.8)
        ax.set_title(title, fontweight="bold")
        ax.set_xlabel(metric)
    plt.suptitle("Per-Page OCR Image Quality Distributions", fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "ocr_quality_distributions.png", dpi=300)
    plt.show()
else:
    print("Skipping quality plots (no PDF data available).")

---
## Part 2: BERT Document Embeddings

Uses `sentence-transformers` (lighter than full TF Hub BERT) to encode each case's extracted text into a fixed-size embedding vector.

In [ ]:
# Install if needed: pip install sentence-transformers
try:
    from sentence_transformers import SentenceTransformer
    BERT_AVAILABLE = True
    print("sentence-transformers loaded.")
except ImportError:
    BERT_AVAILABLE = False
    print("sentence-transformers not installed. Run: pip install sentence-transformers")
    print("Skipping BERT embeddings; text features will still be computed.")

In [ ]:
if BERT_AVAILABLE:
    model = SentenceTransformer("all-mpnet-base-v2")
    print(f"Model loaded: all-mpnet-base-v2 (dim={model.get_sentence_embedding_dimension()})")

    embeddings = {}
    txt_files = sorted(TEXT_DIR.glob("*.txt"))
    for txt_path in tqdm(txt_files, desc="Encoding documents"):
        case_id = txt_path.stem
        text = txt_path.read_text(encoding="utf-8")
        # Truncate to ~512 tokens worth of text (roughly 2000 chars)
        text_trunc = text[:4000]
        emb = model.encode(text_trunc)
        embeddings[case_id] = emb

    # Save as DataFrame
    emb_dim = model.get_sentence_embedding_dimension()
    emb_rows = []
    for case_id, emb in embeddings.items():
        row = {"case_id": case_id}
        for i in range(emb_dim):
            row[f"bert_dim_{i}"] = float(emb[i])
        emb_rows.append(row)
    df_embeddings = pd.DataFrame(emb_rows)
    df_embeddings.to_csv(FEATURE_DIR / "bert_document_embeddings.csv", index=False)
    print(f"Saved BERT embeddings: {df_embeddings.shape}")
else:
    print("BERT embeddings skipped.")

---
## Part 3: Text-Based Document Features

Compute per-case features from extracted text:
- Token count, unique token ratio, lexical diversity
- Negation frequency, uncertainty language frequency
- Embedding variance (if BERT available)
- Sentence count

In [ ]:
NEGATION_WORDS = {"no", "not", "none", "negative", "absent", "without", "deny", "denied",
                   "unremarkable", "normal", "benign"}
UNCERTAINTY_WORDS = {"possible", "possibly", "probable", "probably", "suggest", "suggestive",
                      "suspicious", "cannot exclude", "cannot rule out", "equivocal",
                      "uncertain", "indeterminate", "likely", "unlikely", "may", "might"}

def compute_text_features(text: str) -> Dict:
    tokens = re.findall(r"\b\w+\b", text.lower())
    n_tokens = len(tokens)
    unique_tokens = set(tokens)
    n_unique = len(unique_tokens)
    sentences = re.split(r"[.!?]+", text)
    sentences = [s.strip() for s in sentences if s.strip()]

    neg_count = sum(1 for t in tokens if t in NEGATION_WORDS)
    unc_count = sum(1 for phrase in UNCERTAINTY_WORDS if phrase in text.lower())

    return {
        "n_tokens": n_tokens,
        "n_unique_tokens": n_unique,
        "lexical_diversity": n_unique / n_tokens if n_tokens > 0 else 0.0,
        "n_sentences": len(sentences),
        "avg_sentence_length": n_tokens / len(sentences) if sentences else 0.0,
        "negation_count": neg_count,
        "negation_rate": neg_count / n_tokens if n_tokens > 0 else 0.0,
        "uncertainty_count": unc_count,
        "uncertainty_rate": unc_count / n_tokens if n_tokens > 0 else 0.0,
        "char_count": len(text),
    }

text_feature_rows = []
txt_files = sorted(TEXT_DIR.glob("*.txt"))
for txt_path in tqdm(txt_files, desc="Computing text features"):
    case_id = txt_path.stem
    text = txt_path.read_text(encoding="utf-8")
    feats = compute_text_features(text)
    feats["case_id"] = case_id
    text_feature_rows.append(feats)

df_text_features = pd.DataFrame(text_feature_rows)
df_text_features.to_csv(FEATURE_DIR / "case_text_features.csv", index=False)
print(f"Text features: {df_text_features.shape}")
df_text_features.describe()

In [ ]:
# Embedding variance per case (measure of document complexity)
if BERT_AVAILABLE and len(embeddings) > 0:
    emb_variance = []
    for case_id, emb in embeddings.items():
        emb_variance.append({"case_id": case_id, "embedding_variance": float(np.var(emb))})
    df_emb_var = pd.DataFrame(emb_variance)
    df_text_features = df_text_features.merge(df_emb_var, on="case_id", how="left")
    df_text_features.to_csv(FEATURE_DIR / "case_text_features.csv", index=False)
    print(f"Added embedding variance. Updated shape: {df_text_features.shape}")

# Merge quality + text features
df_all_features = df_text_features.copy()
if "case_id" in case_quality.columns:
    df_all_features = df_all_features.merge(case_quality, on="case_id", how="left")
df_all_features.to_csv(FEATURE_DIR / "case_all_features.csv", index=False)
print(f"Combined feature matrix: {df_all_features.shape}")
df_all_features.head()

---
## Part 4: Feature Interaction Analysis (H2O)

Adapted from `ai_feature_interaction_analysis.py`. Uses H2O XGBoost/GBM to identify which feature combinations predict AI accuracy and fabrication.

In [ ]:
# Check if comprehensive dataset exists for interaction analysis
comp_path_candidates = [
    PROJECT_ROOT / "reports" / "comprehensive_enhanced_dataset_with_all_metrics.csv",
    PROJECT_ROOT / "data" / "processed" / "comprehensive_enhanced_dataset_with_all_metrics.csv",
]
comp_path = None
for p in comp_path_candidates:
    if p.exists():
        comp_path = p
        break

if comp_path:
    df_comp = pd.read_csv(comp_path)
    print(f"Loaded comprehensive dataset: {df_comp.shape} from {comp_path}")
else:
    print("Comprehensive dataset not found. Run create_comprehensive_enhanced_dataset.py first.")
    print("Skipping H2O feature interaction analysis.")
    df_comp = None

In [ ]:
# H2O Feature Interaction Analysis
H2O_AVAILABLE = False
if df_comp is not None:
    try:
        import h2o
        from h2o.estimators import H2OXGBoostEstimator, H2OGradientBoostingEstimator
        H2O_AVAILABLE = True
    except ImportError:
        print("h2o not installed. Run: pip install h2o")

if H2O_AVAILABLE:
    h2o.init(max_mem_size="4G", nthreads=-1)
    print("H2O initialized.")

    # AI correct features
    ai_correct_cols = [c for c in df_comp.columns if c.endswith("_ai_correct")]
    # AI error type columns (for fabrication targets)
    ai_error_cols = [c for c in df_comp.columns if c.endswith("_ai_error_type")]

    print(f"AI correct features: {len(ai_correct_cols)}")
    print(f"AI error type columns: {len(ai_error_cols)}")

    # Create fabrication binary targets
    for col in ai_error_cols:
        element = col.replace("_ai_error_type", "")
        df_comp[f"{element}_ai_fabricated"] = (df_comp[col] == "fabrication").astype(int)

    # Target: any fabrication across elements
    fab_cols = [c for c in df_comp.columns if c.endswith("_ai_fabricated")]
    df_comp["any_ai_fabrication"] = (df_comp[fab_cols].sum(axis=1) > 0).astype(int)

    # Target: total AI correct
    if ai_correct_cols:
        df_comp["ai_total_correct"] = df_comp[ai_correct_cols].sum(axis=1)

    print(f"Fabrication rate: {df_comp['any_ai_fabrication'].mean():.3f}")
else:
    print("Skipping H2O analysis.")

In [ ]:
def train_and_extract_interactions(df, features, target, target_name, model_type="xgboost"):
    """Train a model and extract feature interactions."""
    h2o_df = h2o.H2OFrame(df[features + [target]].dropna())
    h2o_df[target] = h2o_df[target].asfactor() if df[target].nunique() <= 2 else h2o_df[target]

    if model_type == "xgboost":
        model = H2OXGBoostEstimator(
            ntrees=200, max_depth=5, learn_rate=0.1,
            sample_rate=0.8, col_sample_rate=0.8, seed=42
        )
    else:
        model = H2OGradientBoostingEstimator(
            ntrees=200, max_depth=5, learn_rate=0.1, seed=42
        )

    model.train(x=features, y=target, training_frame=h2o_df)

    # Variable importance
    varimp = model.varimp(use_pandas=True)
    varimp["target"] = target_name
    varimp["model_type"] = model_type

    return model, varimp

if H2O_AVAILABLE and ai_correct_cols:
    all_varimp = []

    # Model 1: Predict any_ai_fabrication from AI correct features
    print("\nTraining: AI correct features -> any_ai_fabrication")
    model_fab, varimp_fab = train_and_extract_interactions(
        df_comp, ai_correct_cols, "any_ai_fabrication", "Any Fabrication"
    )
    all_varimp.append(varimp_fab)

    # Model 2: Predict ai_total_correct from AI correct features
    print("Training: AI correct features -> ai_total_correct")
    model_acc, varimp_acc = train_and_extract_interactions(
        df_comp, ai_correct_cols, "ai_total_correct", "Total Correct",
        model_type="gbm"
    )
    all_varimp.append(varimp_acc)

    df_varimp = pd.concat(all_varimp, ignore_index=True)
    df_varimp.to_csv(FEATURE_DIR / "feature_interactions_summary.csv", index=False)
    print(f"\nFeature importance summary: {df_varimp.shape}")
    df_varimp.head(20)
else:
    print("Feature interaction analysis skipped.")

In [ ]:
# Feature importance visualization
if H2O_AVAILABLE and 'df_varimp' in dir() and len(df_varimp) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(16, 8))
    for ax_i, target_name in enumerate(df_varimp["target"].unique()):
        ax = axes[ax_i] if len(df_varimp["target"].unique()) > 1 else axes
        sub = df_varimp[df_varimp["target"] == target_name].head(15)
        ax.barh(range(len(sub)), sub["scaled_importance"].values, color="#3498db")
        ax.set_yticks(range(len(sub)))
        ax.set_yticklabels(sub["variable"].values, fontsize=9)
        ax.set_xlabel("Scaled Importance")
        ax.set_title(f"Top Features: {target_name}", fontsize=13, fontweight="bold")
        ax.invert_yaxis()

    plt.suptitle("Feature Importance: AI Accuracy & Fabrication Predictors",
                 fontsize=15, fontweight="bold")
    plt.tight_layout()
    plt.savefig(OUTPUT_DIR / "feature_importance_interactions.png", dpi=300, bbox_inches="tight")
    plt.show()

    # Shutdown H2O
    h2o.shutdown(prompt=False)
else:
    print("No feature importance to plot.")

---
## Summary of Outputs

| Output | Path |
|--------|------|
| Per-page OCR quality | `data/features/page_level_ocr_quality.csv` |
| Case-level OCR quality | `data/features/case_level_ocr_quality.csv` |
| BERT embeddings | `data/features/bert_document_embeddings.csv` |
| Text features | `data/features/case_text_features.csv` |
| Combined features | `data/features/case_all_features.csv` |
| Feature interactions | `data/features/feature_interactions_summary.csv` |
| Quality distributions plot | `reports/ocr_quality_distributions.png` |
| Feature importance plot | `reports/feature_importance_interactions.png` |

In [ ]:
print("\nAll outputs saved.")
print(f"  Features directory: {FEATURE_DIR}")
print(f"  Reports directory: {OUTPUT_DIR}")
print(f"\nFiles in features/: {list(FEATURE_DIR.glob('*.csv'))}")